# Acceptance: parameterised `split=`

**This is a sign-off page, not a tutorial.** It checks that a flow's `split=` proportions can be
parameters (plan `plans/param-split.plan.md`). The tutorial version is the guide page
*Stratification*, sections "Splitting a flow by a parameter" and "What an adjustment can be".

The case comes from the tb_acf model. Progression from incipient to active TB lands in one of
three `focus` strata, and a parameter `progression_prop_infectious` decides how many become
high-focus. `split=` used to take numbers only, so the model had to write one flow per
destination with complementary rates:

```python
TransitionFlow("progression_low_focus", ..., Param("progression_rate") * (1.0 - Param("progression_prop_infectious")))
TransitionFlow("progression_high_focus", ..., Param("progression_rate") * Param("progression_prop_infectious"))
```

Now one flow says it:

```python
p = Param("progression_prop_infectious")
TransitionFlow("progression", stage["incipient"], stage["active"], Param("progression_rate"),
               split={focus: {"low": 1 - p, "mid": 0.0, "high": p}})
```

A proportion can be a number or any rate expression. Numbers must add up to 1, as before.
Expressions are only known when the model runs, so summer4 uses them as relative weights: each
one is divided by the total for its source compartment. The model is compiled once, and
changing the parameter needs no rebuild.

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import pandas as pd
import plotly.io as pio

pd.options.plotting.backend = "plotly"
pio.renderers.default = "notebook_connected"

from summer4 import (
    Compartments,
    FlowMass,
    FlowModel,
    OutputSet,
    Param,
    Property,
    PropertyMap,
    Time,
    TransitionFlow,
)

stage = Property("stage", ("incipient", "active", "cleared"))
extent = Property("extent", ("low", "high"))
focus = Property("focus", ("low", "mid", "high"))
# As in tb_acf: extent everywhere, focus only on low-extent active disease.
pmap = (
    PropertyMap.from_property(stage)
    .stratify(extent)
    .stratify(focus, where=stage["active"] & extent["low"])
)
rate, p = Param("progression_rate"), Param("progression_prop_infectious")
params = {"progression_rate": 0.1, "progression_prop_infectious": 0.3, "clearance_rate": 0.2}
start = {stage["incipient"] & extent["low"]: 800.0, stage["incipient"] & extent["high"]: 200.0}


def clearance() -> TransitionFlow:
    return TransitionFlow("clearance", stage["active"], stage["cleared"], Param("clearance_rate"))


def with_split(share: object = p) -> FlowModel:
    """One progression flow; `share` of low-extent progression becomes high-focus."""
    model = FlowModel(pmap)
    model.add_flow(
        TransitionFlow(
            "progression",
            stage["incipient"],
            stage["active"],
            rate,
            split={focus: {"low": 1 - share, "mid": 0.0, "high": share}},
        )
    )
    model.add_flow(clearance())
    model.set_initial_population(start)
    return model


def hand_written() -> FlowModel:
    """The tb_acf workaround: one flow per destination."""
    model = FlowModel(pmap)
    low_extent = stage["incipient"] & extent["low"]
    model.add_flow(
        TransitionFlow(
            "progression_low_focus",
            low_extent,
            stage["active"] & extent["low"] & focus["low"],
            rate * (1.0 - p),
        )
    )
    model.add_flow(
        TransitionFlow(
            "progression_high_focus",
            low_extent,
            stage["active"] & extent["low"] & focus["high"],
            rate * p,
        )
    )
    model.add_flow(
        TransitionFlow(
            "progression_high_extent",
            stage["incipient"] & extent["high"],
            stage["active"] & extent["high"],
            rate,
        )
    )
    model.add_flow(clearance())
    model.set_initial_population(start)
    return model


active = OutputSet()
active["active"] = Compartments(where=stage["active"])


def active_people(model: FlowModel) -> pd.DataFrame:
    result = model.compile().run(params, t0=0.0, t1=60.0, dt=1.0, save=active.plan())
    return active.evaluate(result, params)["active"].to_pandas()

## A. One flow with a parameterised split equals the hand-written flows

Both models run with the same parameters. The plot shows the four active compartments from each:
solid lines from the single flow with `split=`, dashed from the three hand-written flows. The
high-extent compartment has no focus strata, so its source sends everyone to that one compartment
in both models.

In [ ]:
split_run = active_people(with_split())
hand_run = active_people(hand_written())
labels = {name: name.replace("stage=active_", "") for name in split_run.columns}
both = split_run.rename(columns=labels).add_suffix(" (split=)").join(
    hand_run.rename(columns=labels).add_suffix(" (three flows)")
)
fig = both.plot(
    title="A. Active TB by compartment: one flow with split= vs three flows",
    labels={"time": "day", "value": "people", "variable": ""},
)
for trace in fig.data[len(labels):]:
    trace.update(line={"dash": "dash"})
fig.show()
pd.Series(
    {
        "progression flows (split=)": 1,
        "progression flows (hand-written)": 3,
        "largest difference, people": float(np.abs(split_run.to_numpy() - hand_run.to_numpy()).max()),
    },
    name="A",
)

> **What to check (A).** Each dashed line lies on the solid line of the same compartment.
> `focus=mid` stays at zero. The largest difference is at rounding level (below 1e-9 people).

## B. The split follows the parameter without recompiling

One compiled model runs at five values of `progression_prop_infectious`. The bars show the first
day's progression into each focus stratum. 800 low-extent people progress at rate 0.1, so 80 per
day are shared between low and high focus.

In [ ]:
compiled = with_split().compile()
progression = OutputSet()
progression["by_focus"] = FlowMass("progression").sum_by(focus, side="dest")

first_day = {}
for value in (0.0, 0.25, 0.5, 0.75, 1.0):
    run_params = {**params, "progression_prop_infectious": value}
    result = compiled.run(run_params, t0=0.0, t1=10.0, dt=1.0, save=progression.plan())
    first_day[value] = progression.evaluate(result, run_params)["by_focus"].to_pandas().iloc[0]
sweep = pd.DataFrame(first_day).T
sweep.index.name = "progression_prop_infectious"
sweep.plot.bar(
    title="B. Progression on day 0, by focus, from one compiled model",
    labels={"value": "people per day", "variable": ""},
    barmode="group",
)

> **What to check (B).** Low focus falls from 80 to 0 as the parameter rises, high focus
> rises from 0 to 80, and they add to 80 in every group. Mid focus is always zero.

## C. A proportion can change over time

A proportion can be any rate expression, including a function of time. Here the high-focus share
is `Time() / 60`, so it grows from 0 at day 0 to 1 at day 60. The plot shows the share of
low-extent progression that went to high focus on each day, beside the straight line `t / 60`.

In [ ]:
timed = with_split(share=Time() / 60.0).compile()
result = timed.run(params, t0=0.0, t1=60.0, dt=1.0, save=progression.plan())
flows = progression.evaluate(result, params)["by_focus"].to_pandas()
share = pd.DataFrame(
    {
        "high-focus share of progression": flows["focus=high"] / (flows["focus=low"] + flows["focus=high"]),
        "t / 60": flows.index.to_numpy() / 60.0,
    }
)
fig = share.plot(
    title="C. High-focus share with split={focus: {..., 'high': Time() / 60}}",
    labels={"time": "day", "value": "share", "variable": ""},
)
fig.data[1].update(line={"dash": "dash"})
fig

> **What to check (C).** The share rises in a straight line from 0 to 1, on the dashed line.

## D. Gradients pass through the split

Calibration needs derivatives with respect to the split parameter. The plot compares `jax.grad`
of high-focus active people on day 30 with a central finite difference, at several parameter
values.

In [ ]:
high = OutputSet()
high["high"] = Compartments(where=stage["active"] & focus["high"]).sum_by()


def high_on_day_30(value: jax.Array) -> jax.Array:
    run_params = {**params, "progression_prop_infectious": value}
    result = compiled.run(run_params, t0=0.0, t1=30.0, dt=1.0, save=high.plan())
    return jnp.asarray(high.evaluate(result, run_params)["high"].values)[-1]


grad = jax.jit(jax.grad(high_on_day_30))
step = 1e-4
points = np.linspace(0.1, 0.9, 5)
gradients = pd.DataFrame(
    {
        "jax.grad": [float(grad(jnp.asarray(v))) for v in points],
        "finite difference": [
            float((high_on_day_30(v + step) - high_on_day_30(v - step)) / (2 * step)) for v in points
        ],
    },
    index=pd.Index(points, name="progression_prop_infectious"),
)
fig = gradients.plot(
    title="D. d(high-focus active on day 30) / d(progression_prop_infectious)",
    labels={"value": "people per unit of the parameter", "variable": ""},
    markers=True,
)
fig.data[1].update(line={"dash": "dash"})
fig

> **What to check (D).** The two lines lie on top of each other. The derivative is the same at
> every point, because high-focus people are proportional to the parameter.